# Datasets and Dataset Attachments with the ML App Python client

A focused, rerunnable walkthrough of dataset CRUD, immutable versions, and role-specific Dataset-to-Business-Case mappings. It uses bounded pages for display and streams a local CSV from disk.

In [7]:
from pathlib import Path

from ml_app_client import MLAppClient, ResourceNotFoundError, errors

client = MLAppClient.connect()
profile = client.me()
print(f"Connected as {profile['login_name']}")

Connected as test3@example.pl


In [8]:
DATASET_NAME = "Client module example churn source dataset"
DATASET_PATH = Path("../data/general-example.csv")

if not DATASET_PATH.is_file():
    raise FileNotFoundError(DATASET_PATH)

## Create or read a dataset

In [ ]:
churn = client.upload_dataset(DATASET_PATH, name=DATASET_NAME)

In [11]:
dataset = client.get_dataset("f218645b-68a4-4fe0-a62c-1f15564b031a")
sink = client.display_object(dataset)

Id,f218645b-68a4-4fe0-a62c-1f15564b031a
Logical Id,60bd9f1f-451a-4638-a828-a7b0d2626992
Name,Client module example churn source dataset
Version Number,1
Row Count,10000
Format,csv
Owner Id,c3945744-aa0b-4d94-8c84-24a7d46fbe8c
Source Type,file
Version Stage,source
Description,—
Status,ready


In [13]:
sink = client.display_dataset_metadata(dataset)

In [35]:
client.preview_dataset(dataset.id, limit=20)

{'dataset_id': '5cf0f4b2-e0aa-4ba2-a5e9-aafc8a2de118',
 'columns': [{'name': 'customer_id', 'type': 'text'},
  {'name': 'snapshot_month', 'type': 'text'},
  {'name': 'region', 'type': 'text'},
  {'name': 'acquisition_channel', 'type': 'text'},
  {'name': 'customer_segment', 'type': 'text'},
  {'name': 'plan_type', 'type': 'text'},
  {'name': 'age', 'type': 'number'},
  {'name': 'tenure_months', 'type': 'number'},
  {'name': 'household_income', 'type': 'number'},
  {'name': 'credit_score', 'type': 'number'},
  {'name': 'monthly_fee', 'type': 'number'},
  {'name': 'avg_monthly_usage_gb', 'type': 'number'},
  {'name': 'app_sessions_30d', 'type': 'number'},
  {'name': 'support_tickets_90d', 'type': 'number'},
  {'name': 'late_payments_12m', 'type': 'number'},
  {'name': 'discount_pct', 'type': 'number'},
  {'name': 'data_overage_charges', 'type': 'number'},
  {'name': 'competitor_price_index', 'type': 'number'},
  {'name': 'nps_score', 'type': 'number'},
  {'name': 'churned', 'type': 'numb

In [36]:
path = client.download_dataset(dataset.id, "./test.csv")

## Update metadata, not immutable content

The REST contract merges metadata. Changing the data itself means uploading a distinct version.

In [14]:
dataset = client.update_dataset_metadata(
    dataset,
    metadata={"Custom Metadata Example": {"Key": "Value", "reviewed": True}},
)
print(dataset.metadata.get("Custom Metadata Example"))
sink = client.display_dataset_metadata(dataset)


{'Key': 'Value', 'reviewed': True}


## Catalogue of datasets

In [27]:
page = client.page_datasets(limit=3)
for it, dataset in enumerate(page.items):
    sink = client.display_object(dataset)

Id,f89908f1-6a88-570b-8276-c28f9c285a60
Logical Id,f89908f1-6a88-570b-8276-c28f9c285a60
Name,Example01 10 - Estates Model Service - demo predictions with actuals 2026-07-21 to 2026-07-22
Version Number,1
Row Count,4
Format,parquet
Owner Id,c3945744-aa0b-4d94-8c84-24a7d46fbe8c
Source Type,file
Version Stage,final
Description,Immutable full-scope online monitoring materialization
Status,ready


Id,1f3e189c-78b4-597a-9592-d6e22804002b
Logical Id,1f3e189c-78b4-597a-9592-d6e22804002b
Name,Example01 10 - Estates Model Service - demo online predictions 2026-07-21 to 2026-07-22
Version Number,1
Row Count,4
Format,parquet
Owner Id,c3945744-aa0b-4d94-8c84-24a7d46fbe8c
Source Type,file
Version Stage,final
Description,Immutable full-scope online monitoring materialization
Status,ready


Id,23758d26-ab44-5223-bf5a-4de2c0ea9d23
Logical Id,e313d13e-fd6a-53b0-8acd-d8af370af7b3
Name,Example01 Estates AutoML - Predictions with Actuals
Version Number,1
Row Count,100000
Format,parquet
Owner Id,c3945744-aa0b-4d94-8c84-24a7d46fbe8c
Source Type,file
Version Stage,intermediate
Description,Output of pipeline run 4d3c1189-c278-4360-a2b1-92b272f62b66
Status,ready


## Soft-delete a dataset

In [37]:
client.delete_dataset(dataset)

Dataset(id='5cf0f4b2-e0aa-4ba2-a5e9-aafc8a2de118', logical_id='fdb50b35-555f-4cda-9c1d-05f3565c0d8c', name='Client module example churn source dataset', version_number=1, row_count=10000, format='csv', owner_id='c3945744-aa0b-4d94-8c84-24a7d46fbe8c', source_type='file', version_stage='source', description='', status='deleted', tags=(), metadata={'source_schema': [{'name': 'customer_id', 'type': 'text'}, {'name': 'snapshot_month', 'type': 'text'}, {'name': 'region', 'type': 'text'}, {'name': 'acquisition_channel', 'type': 'text'}, {'name': 'customer_segment', 'type': 'text'}, {'name': 'plan_type', 'type': 'text'}, {'name': 'age', 'type': 'number'}, {'name': 'tenure_months', 'type': 'number'}, {'name': 'household_income', 'type': 'number'}, {'name': 'credit_score', 'type': 'number'}, {'name': 'monthly_fee', 'type': 'number'}, {'name': 'avg_monthly_usage_gb', 'type': 'number'}, {'name': 'app_sessions_30d', 'type': 'number'}, {'name': 'support_tickets_90d', 'type': 'number'}, {'name': 'lat

## Attach a dataset into a business case

In [39]:
BUSINESS_CASE_NAME = "[MLAPP client module] Customer churn demo"
case, created = client.ensure_business_case(name=BUSINESS_CASE_NAME, problem_type="binary_classification")
display(created)

False

In [40]:
dataset = client.upload_dataset(DATASET_PATH, name=DATASET_NAME)
sink = client.display_object(dataset)

Id,234d9fa4-6591-4767-809d-6f2159c85198
Logical Id,883494df-46dd-4fc3-9899-9b7e5339e5f0
Name,Client module example churn source dataset
Version Number,1
Row Count,10000
Format,csv
Owner Id,c3945744-aa0b-4d94-8c84-24a7d46fbe8c
Source Type,file
Version Stage,source
Description,—
Status,ready


In [41]:
attachment = client.create_dataset_attachment(
    case, 
    dataset, 
    role="source", 
    primary_key_column="customer_id", 
    target_column="churned"
)
sink = client.display_object(attachment)

Id,d57696fd-faa0-449e-899c-75c23ed93ded
Business Case Id,413a4d16-235f-48d7-957b-cc02592e7ef0
Data Asset Id,234d9fa4-6591-4767-809d-6f2159c85198
Role,source
Context Note,—


In [45]:
attachment = client.update_dataset_attachment(case, attachment, context_note="An input data set to the churn analysis")

In [ ]:
for item in client.iter_dataset_attachments(case): 
    sink = client.display_object(item)

Id,d57696fd-faa0-449e-899c-75c23ed93ded
Business Case Id,413a4d16-235f-48d7-957b-cc02592e7ef0
Data Asset Id,234d9fa4-6591-4767-809d-6f2159c85198
Role,source
Context Note,An input data set to the churn analysis


In [47]:
client.delete_dataset_attachment(case, attachment)

## Adding a new version to the dataset

In [48]:
BUSINESS_CASE_NAME = "[MLAPP client module] Customer churn demo"
case, created = client.ensure_business_case(name=BUSINESS_CASE_NAME, problem_type="binary_classification")
display(created)

False

In [51]:
dataset, created = client.ensure_dataset(
    DATASET_PATH, 
    business_case_name=BUSINESS_CASE_NAME, 
    dataset_name=DATASET_NAME, 
    role="source",
    force=True
)
print(created)

True


In [52]:
for i in range(3):
    version = client.upload_dataset_version(DATASET_PATH, business_case_name=BUSINESS_CASE_NAME, dataset_name=DATASET_NAME)